# Customer Churn Prediction\n\nThis notebook trains multiple classification models to predict customer churn and compares their performance.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


In [ ]:
df = pd.read_csv('data/customer_churn.csv')
df.head()

In [ ]:
target = 'Churn'
df = df.dropna(subset=[target])
X = df.drop(columns=[target])
y = df[target]

if y.dtype == 'object':
    y = y.astype(str).str.lower().map({'yes': 1, 'true': 1, 'churn': 1, '1': 1, 'no': 0, 'false': 0, 'not churn': 0, '0': 0})
y = y.astype(int)

cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.columns.difference(cat_cols)

numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
    ,('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(
    transformers=[('num', numeric_transformer, num_cols), ('cat', categorical_transformer, cat_cols)]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'Logistic Regression': LogisticRegression(max_iter=200),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = {}
for name, clf in models.items():
    pipeline = Pipeline(steps=[('preprocess', preprocess), ('model', clf)])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    proba = pipeline.predict_proba(X_test)[:, 1]

    results[name] = {
        'accuracy': accuracy_score(y_test, preds),
        'f1': f1_score(y_test, preds),
        'roc_auc': roc_auc_score(y_test, proba)
    }

results